In [2]:
import numpy as np
from collections import Counter
from itertools import combinations, permutations
from ase.build import surface, make_supercell
from ase.io import write, read
from ase.visualize import view
from ase import Atom
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.io.lammps.data import LammpsData
from pymatgen.core.surface import Slab
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
import os

os.makedirs("slabs", exist_ok=True)
alloy = read('bulk_structure/Mg4Zn7_relaxed.cif')
print(f"Bulk: {len(alloy)} atoms, {alloy.get_chemical_formula()}")

Bulk: 110 atoms, Mg40Zn70


In [3]:
slab = surface(alloy, (1, 0, 1), 2, vacuum=8)

sym = np.array(slab.get_chemical_symbols())
mg, zn = sum(sym == 'Mg'), sum(sym == 'Zn')
z = slab.get_positions()[:, 2]
thick = z.max() - z.min()

ps = AseAtomsAdaptor().get_structure(slab)
slab_obj = Slab(ps.lattice, ps.species, ps.frac_coords,
    miller_index=(1,0,1), oriented_unit_cell=ps, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

print(f"Atoms: {len(slab)}, Mg: {mg}, Zn: {zn}")
print(f"Thickness: {thick:.1f} A")
print(f"Ratio Zn/Mg: {zn/mg:.4f} (need 1.75)")
print(f"Stoichiometric: {'YES' if zn*4 == mg*7 else 'NO'}")
print(f"Symmetric: {slab_obj.is_symmetric()}")

Atoms: 220, Mg: 80, Zn: 140
Thickness: 22.6 A
Ratio Zn/Mg: 1.7500 (need 1.75)
Stoichiometric: YES
Symmetric: False


In [4]:
z = slab.get_positions()[:, 2]
sym = np.array(slab.get_chemical_symbols())
order = np.argsort(z)

# Check z-spacing to pick correct tolerance
z_sorted = np.sort(z)
gaps = np.diff(z_sorted)
print(f"Min gap: {gaps.min():.4f} A, Median gap: {np.median(gaps):.4f} A")
tol = max(0.02, gaps.min() / 2)
print(f"Using tolerance: {tol:.3f} A\n")

layers = []
layer_idx = []
cur = [order[0]]
for i in range(1, len(order)):
    if z[order[i]] - z[order[i-1]] < tol:
        cur.append(order[i])
    else:
        layers.append(Counter(sym[j] for j in cur))
        layer_idx.append(list(cur))
        cur = [order[i]]
layers.append(Counter(sym[j] for j in cur))
layer_idx.append(list(cur))

z_min, z_max = z.min(), z.max()
n = len(layers)

print(f"Total atomic layers: {n}\n")
print(f"{'Bot':>7} {'Bot Comp':>10} {'from_bot':>9}  |  {'Top':>7} {'Top Comp':>10} {'from_top':>9} {'Match':>6}")
print("-" * 80)
for i in range(min(15, n//2)):
    bot, top = layers[i], layers[n-1-i]
    zm_b = np.mean([z[j] for j in layer_idx[i]])
    zm_t = np.mean([z[j] for j in layer_idx[n-1-i]])
    
    mg_b, zn_b = bot.get('Mg',0), bot.get('Zn',0)
    mg_t, zn_t = top.get('Mg',0), top.get('Zn',0)
    comp_b = f"Mg{mg_b}Zn{zn_b}" if mg_b and zn_b else (f"Mg{mg_b}" if mg_b else f"Zn{zn_b}")
    comp_t = f"Mg{mg_t}Zn{zn_t}" if mg_t and zn_t else (f"Mg{mg_t}" if mg_t else f"Zn{zn_t}")
    
    match = "YES" if bot == top else "NO <---"
    print(f"{i:>7} {comp_b:>10} {zm_b-z_min:>9.3f}  |  {n-1-i:>7} {comp_t:>10} {z_max-zm_t:>9.3f} {match:>6}")

Min gap: 0.0000 A, Median gap: 0.0532 A
Using tolerance: 0.020 A

Total atomic layers: 144

    Bot   Bot Comp  from_bot  |      Top   Top Comp  from_top  Match
--------------------------------------------------------------------------------
      0        Zn1     0.000  |      143        Zn2     0.000 NO <---
      1        Zn1     0.390  |      142     Mg2Zn1     0.046 NO <---
      2        Mg1     0.557  |      141        Zn2     0.092 NO <---
      3        Zn1     0.718  |      140        Zn1     0.216    YES
      4        Mg1     0.782  |      139        Zn1     0.606 NO <---
      5        Mg1     0.835  |      138        Mg1     0.773    YES
      6     Mg1Zn2     0.883  |      137        Zn1     0.934 NO <---
      7        Mg1     0.990  |      136        Mg1     0.998    YES
      8        Zn1     1.408  |      135        Mg1     1.052 NO <---
      9        Mg1     1.598  |      134     Mg1Zn2     1.100 NO <---
     10        Zn3     1.639  |      133        Mg1     1.206

In [5]:
print("Searching for removals that give Symmetric=True...\n")

found_any = False
for bot_rm in range(0, 12):
    for top_rm in range(0, 12):
        if bot_rm == 0 and top_rm == 0:
            continue
        
        keep = []
        for j in range(bot_rm, n - top_rm):
            keep.extend(layer_idx[j])
        if len(keep) == 0:
            continue
        
        tr = slab[keep]
        try:
            ps_tr = AseAtomsAdaptor().get_structure(tr)
            slab_tr = Slab(ps_tr.lattice, ps_tr.species, ps_tr.frac_coords,
                miller_index=(1,0,1), oriented_unit_cell=ps_tr, shift=0,
                scale_factor=[[1,0,0],[0,1,0],[0,0,1]])
            is_sym = slab_tr.is_symmetric()
        except:
            is_sym = False
        
        if is_sym:
            sym_tr = np.array(tr.get_chemical_symbols())
            mg_tr = sum(sym_tr == 'Mg')
            zn_tr = sum(sym_tr == 'Zn')
            removed_bot = sum(len(layer_idx[j]) for j in range(bot_rm))
            removed_top = sum(len(layer_idx[j]) for j in range(n - top_rm, n))
            print(f"  bot_rm={bot_rm}, top_rm={top_rm}: "
                  f"{len(tr)} atoms, Mg{mg_tr} Zn{zn_tr}, "
                  f"removed {removed_bot}+{removed_top}={removed_bot+removed_top}")
            found_any = True

if not found_any:
    print("No symmetric sub-slab found with up to 11 layers removed from each end")

Searching for removals that give Symmetric=True...

  bot_rm=0, top_rm=3: 213 atoms, Mg78 Zn135, removed 0+7=7
  bot_rm=1, top_rm=4: 211 atoms, Mg78 Zn133, removed 1+8=9
  bot_rm=2, top_rm=5: 209 atoms, Mg78 Zn131, removed 2+9=11
  bot_rm=3, top_rm=6: 207 atoms, Mg76 Zn131, removed 3+10=13
  bot_rm=4, top_rm=7: 205 atoms, Mg76 Zn129, removed 4+11=15
  bot_rm=5, top_rm=8: 203 atoms, Mg74 Zn129, removed 5+12=17
  bot_rm=6, top_rm=9: 201 atoms, Mg72 Zn129, removed 6+13=19
  bot_rm=7, top_rm=10: 195 atoms, Mg70 Zn125, removed 9+16=25
  bot_rm=8, top_rm=11: 193 atoms, Mg68 Zn125, removed 10+17=27


In [6]:
removed_top = []
for j in range(n - 3, n):
    removed_top.extend(layer_idx[j])

keep = []
for j in range(0, n - 3):
    keep.extend(layer_idx[j])

trimmed = slab[keep]
ps_trim = AseAtomsAdaptor().get_structure(trimmed)

sga = SpacegroupAnalyzer(ps_trim, symprec=0.1)
print(f"SG = {sga.get_space_group_symbol()}")

for op in sga.get_symmetry_operations():
    if op.rotation_matrix[2][2] < 0:
        inv_translation = op.translation_vector
        print(f"Inversion: f -> {inv_translation} - f")
        break

rem_mg = sum(sym[j] == 'Mg' for j in removed_top)
rem_zn = sum(sym[j] == 'Zn' for j in removed_top)
print(f"\nRemoved ({len(removed_top)} atoms): Mg{rem_mg} Zn{rem_zn}")

for j in removed_top:
    pos = slab.positions[j]
    print(f"  idx={j} {sym[j]} ({pos[0]:.3f}, {pos[1]:.3f}, {pos[2]:.3f})")

# Check partners
ps_orig = AseAtomsAdaptor().get_structure(slab)
keep_set = set(keep)
unpaired = []
paired = []
for j in removed_top:
    frac = ps_orig[j].frac_coords
    inv_frac = (inv_translation - frac) % 1.0
    inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
    dists = np.linalg.norm(slab.get_positions() - inv_cart, axis=1)
    min_d = min(dists[k] for k in keep_set)
    closest = min(keep_set, key=lambda k: dists[k])
    if min_d < 0.5:
        paired.append(j)
        print(f"  idx={j} -> paired with idx={closest} dist={min_d:.3f}")
    else:
        unpaired.append(j)
        print(f"  idx={j} -> UNPAIRED, inv at ({inv_cart[0]:.3f}, {inv_cart[1]:.3f}, {inv_cart[2]:.3f})")

print(f"\nPaired: {len(paired)}, Unpaired: {len(unpaired)}")

SG = P2/m
Inversion: f -> [0.44599303 0.         0.99439776] - f

Removed (7 atoms): Mg2 Zn5
  idx=205 Zn (11.255, 3.987, 30.537)
  idx=204 Zn (11.255, 1.253, 30.537)
  idx=116 Mg (25.271, 2.620, 30.575)
  idx=151 Zn (31.060, 2.620, 30.582)
  idx=117 Mg (4.166, 2.620, 30.590)
  idx=206 Zn (18.181, 3.987, 30.628)
  idx=207 Zn (18.181, 1.253, 30.628)
  idx=205 -> UNPAIRED, inv at (3.321, 1.349, 7.875)
  idx=204 -> UNPAIRED, inv at (3.321, 4.084, 7.875)
  idx=116 -> UNPAIRED, inv at (21.990, 2.717, 7.837)
  idx=151 -> UNPAIRED, inv at (16.201, 2.717, 7.830)
  idx=117 -> UNPAIRED, inv at (10.411, 2.717, 7.822)
  idx=206 -> UNPAIRED, inv at (29.080, 1.349, 7.784)
  idx=207 -> UNPAIRED, inv at (29.080, 4.084, 7.784)

Paired: 0, Unpaired: 7


In [7]:
slab_2y = make_supercell(slab, [[1,0,0],[0,2,0],[0,0,1]])

sym2 = np.array(slab_2y.get_chemical_symbols())
mg2, zn2 = sum(sym2 == 'Mg'), sum(sym2 == 'Zn')
z2 = slab_2y.get_positions()[:, 2]

ps2 = AseAtomsAdaptor().get_structure(slab_2y)
slab_obj2 = Slab(ps2.lattice, ps2.species, ps2.frac_coords,
    miller_index=(1,0,1), oriented_unit_cell=ps2, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

print(f"1x2 supercell:")
print(f"Atoms: {len(slab_2y)}, Mg: {mg2}, Zn: {zn2}")
print(f"Stoichiometric: {'YES' if zn2*4 == mg2*7 else 'NO'}")
print(f"Symmetric: {slab_obj2.is_symmetric()}")

# Find symmetric removal
order2 = np.argsort(z2)
tol = 0.02
layers2 = []
layer_idx2 = []
cur = [order2[0]]
for i in range(1, len(order2)):
    if z2[order2[i]] - z2[order2[i-1]] < tol:
        cur.append(order2[i])
    else:
        layers2.append(Counter(sym2[j] for j in cur))
        layer_idx2.append(list(cur))
        cur = [order2[i]]
layers2.append(Counter(sym2[j] for j in cur))
layer_idx2.append(list(cur))

n2 = len(layers2)
print(f"Atomic layers: {n2}")

# Search removals
for bot_rm in range(0, 8):
    for top_rm in range(0, 8):
        if bot_rm == 0 and top_rm == 0:
            continue
        keep2 = []
        for j in range(bot_rm, n2 - top_rm):
            keep2.extend(layer_idx2[j])
        if len(keep2) == 0:
            continue
        tr2 = slab_2y[keep2]
        try:
            ps_tr2 = AseAtomsAdaptor().get_structure(tr2)
            slab_tr2 = Slab(ps_tr2.lattice, ps_tr2.species, ps_tr2.frac_coords,
                miller_index=(1,0,1), oriented_unit_cell=ps_tr2, shift=0,
                scale_factor=[[1,0,0],[0,1,0],[0,0,1]])
            if slab_tr2.is_symmetric():
                removed_bot = sum(len(layer_idx2[j]) for j in range(bot_rm))
                removed_top = sum(len(layer_idx2[j]) for j in range(n2 - top_rm, n2))
                print(f"  bot_rm={bot_rm}, top_rm={top_rm}: {len(tr2)} atoms, "
                      f"removed {removed_bot}+{removed_top}={removed_bot+removed_top}")
                break
        except:
            continue
    else:
        continue
    break

# Get inversion
sga2 = SpacegroupAnalyzer(ps_tr2, symprec=0.1)
print(f"SG = {sga2.get_space_group_symbol()}")
for op in sga2.get_symmetry_operations():
    if op.rotation_matrix[2][2] < 0:
        inv_translation = op.translation_vector
        print(f"Inversion: f -> {inv_translation} - f")
        break

# Get removed atoms
removed_top2 = []
for j in range(n2 - top_rm, n2):
    removed_top2.extend(layer_idx2[j])

rem_mg2 = sum(sym2[j] == 'Mg' for j in removed_top2)
rem_zn2 = sum(sym2[j] == 'Zn' for j in removed_top2)
print(f"\nRemoved: {len(removed_top2)} atoms (Mg{rem_mg2} Zn{rem_zn2})")

1x2 supercell:
Atoms: 440, Mg: 160, Zn: 280
Stoichiometric: YES
Symmetric: False
Atomic layers: 144
  bot_rm=0, top_rm=3: 426 atoms, removed 0+14=14
SG = P2/m
Inversion: f -> [0.44599303 0.         0.99439776] - f

Removed: 14 atoms (Mg4 Zn10)


In [8]:
# Show removed atoms
for j in removed_top2:
    pos = slab_2y.positions[j]
    print(f"  idx={j} {sym2[j]} ({pos[0]:.3f}, {pos[1]:.3f}, {pos[2]:.3f})")

# Auto-pair by matching x,z (offset in y by cell_b/2)
ps_orig2 = AseAtomsAdaptor().get_structure(slab_2y)
used = set()
pairs = []
for i, j1 in enumerate(removed_top2):
    if j1 in used:
        continue
    pos1 = slab_2y.positions[j1]
    for j2 in removed_top2[i+1:]:
        if j2 in used:
            continue
        pos2 = slab_2y.positions[j2]
        if (sym2[j1] == sym2[j2] and 
            abs(pos1[0] - pos2[0]) < 0.1 and 
            abs(pos1[2] - pos2[2]) < 0.1):
            pairs.append((j1, j2))
            used.add(j1)
            used.add(j2)
            print(f"  Pair: ({j1}, {j2}) {sym2[j1]}")
            break

print(f"\nPairs found: {len(pairs)}")

if len(pairs) == 7:
    # Reconstruct
    sc_fixed = slab_2y.copy()
    for keep_idx, move_idx in pairs:
        frac = ps_orig2[keep_idx].frac_coords
        inv_frac = (inv_translation - frac) % 1.0
        inv_cart = ps_orig2.lattice.get_cartesian_coords(inv_frac)
        sc_fixed.positions[move_idx] = inv_cart
        print(f"  Moved {move_idx} -> inv({keep_idx})")

    sym_f = np.array(sc_fixed.get_chemical_symbols())
    mg_f = sum(sym_f == 'Mg')
    zn_f = sum(sym_f == 'Zn')

    ps_fixed = AseAtomsAdaptor().get_structure(sc_fixed)
    slab_fixed = Slab(ps_fixed.lattice, ps_fixed.species, ps_fixed.frac_coords,
        miller_index=(1,0,1), oriented_unit_cell=ps_fixed, shift=0,
        scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

    print(f"\nAtoms: {len(sc_fixed)}, Mg: {mg_f}, Zn: {zn_f}")
    print(f"Stoichiometric: {'YES' if zn_f*4 == mg_f*7 else 'NO'}")
    print(f"Symmetric: {slab_fixed.is_symmetric()}")
else:
    print("Pairing by x,z failed. Trying brute force...")

  idx=205 Zn (11.255, 3.987, 30.537)
  idx=204 Zn (11.255, 1.253, 30.537)
  idx=425 Zn (11.255, 9.324, 30.537)
  idx=424 Zn (11.255, 6.589, 30.537)
  idx=116 Mg (25.271, 2.620, 30.575)
  idx=336 Mg (25.271, 7.957, 30.575)
  idx=371 Zn (31.060, 7.957, 30.582)
  idx=151 Zn (31.060, 2.620, 30.582)
  idx=117 Mg (4.166, 2.620, 30.590)
  idx=337 Mg (4.166, 7.957, 30.590)
  idx=426 Zn (18.181, 9.324, 30.628)
  idx=206 Zn (18.181, 3.987, 30.628)
  idx=207 Zn (18.181, 1.253, 30.628)
  idx=427 Zn (18.181, 6.589, 30.628)
  Pair: (205, 204) Zn
  Pair: (425, 424) Zn
  Pair: (116, 336) Mg
  Pair: (371, 151) Zn
  Pair: (117, 337) Mg
  Pair: (426, 206) Zn
  Pair: (207, 427) Zn

Pairs found: 7
  Moved 204 -> inv(205)
  Moved 424 -> inv(425)
  Moved 336 -> inv(116)
  Moved 151 -> inv(371)
  Moved 337 -> inv(117)
  Moved 206 -> inv(426)
  Moved 427 -> inv(207)

Atoms: 440, Mg: 160, Zn: 280
Stoichiometric: YES
Symmetric: True


In [9]:
view(sc_fixed)

<Popen: returncode: None args: ['C:\\Users\\Kai Savage\\anaconda3\\python.ex...>

In [10]:
thick = sc_fixed.get_positions()[:,2].max() - sc_fixed.get_positions()[:,2].min()
area = np.linalg.norm(np.cross(sc_fixed.cell[0], sc_fixed.cell[1]))

ps = AseAtomsAdaptor().get_structure(sc_fixed)
ld = LammpsData.from_structure(ps, atom_style="atomic")
ld.write_file("slabs/mg4zn7_101_2layers_reconstructed_ase.data")

print(f"Saved: slabs/mg4zn7_101_2layers_reconstructed_ase.data")
print(f"  Atoms: {len(sc_fixed)}")
print(f"  Thickness: {thick:.1f} A")
print(f"  Area: {area:.1f} A²")
print(f"  Stoichiometric: YES")
print(f"  Symmetric: TRUE")

Saved: slabs/mg4zn7_101_2layers_reconstructed_ase.data
  Atoms: 440
  Thickness: 22.8 A
  Area: 348.8 A²
  Stoichiometric: YES
  Symmetric: TRUE


In [11]:
#unrelaxed surface energy calculation
E_slab =  -533.74796      # eV
E_bulk_per_fu =  -141.72792 / 10  # eV per formula unit 
n = 440 / 11                 # formula units in slab = 32
A = 348.8            # Ang^2

S_eV_per_ang2 = (E_slab - n * E_bulk_per_fu) / (2 * A)

# Convert to J/m^2 (1 eV/Ang^2 = 16.0218 J/m^2)
S_J_per_m2 = S_eV_per_ang2 * 16.0218

print(f"E_bulk per formula unit = {E_bulk_per_fu:.6f} eV")
print(f"n * E_bulk = {n * E_bulk_per_fu:.5f} eV")
print(f"E_slab - n*E_bulk = {E_slab - n * E_bulk_per_fu:.5f} eV")
print(f"S = {S_eV_per_ang2:.6f} eV/Ang^2")
print(f"S = {S_J_per_m2:.4f} J/m^2")

E_bulk per formula unit = -14.172792 eV
n * E_bulk = -566.91168 eV
E_slab - n*E_bulk = 33.16372 eV
S = 0.047540 eV/Ang^2
S = 0.7617 J/m^2


In [13]:
#relaxed surface energy calculation
E_slab_relaxed =  -538.47887296465  # eV
E_bulk_per_fu = -141.72792 / 10  # eV per formula unit
n = 440 / 11                      # 32 formula units
A = 348.8               # Ang^2

S_relaxed_eV = (E_slab_relaxed - n * E_bulk_per_fu) / (2 * A)
S_relaxed_Jm2 = S_relaxed_eV * 16.0218

print(f"E_slab_relaxed - n*E_bulk = {E_slab_relaxed - n * E_bulk_per_fu:.5f} eV")
print(f"S relaxed = {S_relaxed_eV:.6f} eV/Ang^2")
print(f"S relaxed = {S_relaxed_Jm2:.4f} J/m^2")

# Compare with unrelaxed
S_unrelaxed_Jm2 = 0.7617
print(f"\nUnrelaxed S = {S_unrelaxed_Jm2:.4f} J/m^2")
print(f"Relaxed S   = {S_relaxed_Jm2:.4f} J/m^2")
print(f"Relaxation energy = {S_unrelaxed_Jm2 - S_relaxed_Jm2:.4f} J/m^2")

E_slab_relaxed - n*E_bulk = 28.43281 eV
S relaxed = 0.040758 eV/Ang^2
S relaxed = 0.6530 J/m^2

Unrelaxed S = 0.7617 J/m^2
Relaxed S   = 0.6530 J/m^2
Relaxation energy = 0.1087 J/m^2


In [15]:
slab4 = surface(alloy, (1, 0, 1), 4, vacuum=8)
slab4_2y = make_supercell(slab4, [[1,0,0],[0,2,0],[0,0,1]])

z = slab4_2y.get_positions()[:, 2]
sym = np.array(slab4_2y.get_chemical_symbols())
order = np.argsort(z)
print(f"Atoms: {len(slab4_2y)}, Thickness: {z.max()-z.min():.1f} A")

tol = 0.02
layers = []
layer_idx = []
cur = [order[0]]
for i in range(1, len(order)):
    if z[order[i]] - z[order[i-1]] < tol:
        cur.append(order[i])
    else:
        layers.append(Counter(sym[j] for j in cur))
        layer_idx.append(list(cur))
        cur = [order[i]]
layers.append(Counter(sym[j] for j in cur))
layer_idx.append(list(cur))

n = len(layers)
print(f"Atomic layers: {n}")

# Find symmetric removal
for bot_rm in range(0, 8):
    for top_rm in range(0, 8):
        if bot_rm == 0 and top_rm == 0:
            continue
        keep = []
        for j in range(bot_rm, n - top_rm):
            keep.extend(layer_idx[j])
        if len(keep) == 0:
            continue
        tr = slab4_2y[keep]
        try:
            ps_tr = AseAtomsAdaptor().get_structure(tr)
            slab_tr = Slab(ps_tr.lattice, ps_tr.species, ps_tr.frac_coords,
                miller_index=(1,0,1), oriented_unit_cell=ps_tr, shift=0,
                scale_factor=[[1,0,0],[0,1,0],[0,0,1]])
            if slab_tr.is_symmetric():
                removed_bot = sum(len(layer_idx[j]) for j in range(bot_rm))
                removed_top = sum(len(layer_idx[j]) for j in range(n - top_rm, n))
                print(f"  bot_rm={bot_rm}, top_rm={top_rm}: {len(tr)} atoms, "
                      f"removed {removed_bot}+{removed_top}")
                break
        except:
            continue
    else:
        continue
    break

# Get inversion
sga = SpacegroupAnalyzer(ps_tr, symprec=0.1)
print(f"SG = {sga.get_space_group_symbol()}")
for op in sga.get_symmetry_operations():
    if op.rotation_matrix[2][2] < 0:
        inv_translation = op.translation_vector
        print(f"Inversion: f -> {inv_translation} - f")
        break

# Get removed atoms
removed_top_idx = []
for j in range(n - top_rm, n):
    removed_top_idx.extend(layer_idx[j])

print(f"\nRemoved: {len(removed_top_idx)} atoms")

# Auto-pair
ps_orig = AseAtomsAdaptor().get_structure(slab4_2y)
used = set()
pairs = []
for i, j1 in enumerate(removed_top_idx):
    if j1 in used:
        continue
    pos1 = slab4_2y.positions[j1]
    for j2 in removed_top_idx[i+1:]:
        if j2 in used:
            continue
        pos2 = slab4_2y.positions[j2]
        if (sym[j1] == sym[j2] and
            abs(pos1[0] - pos2[0]) < 0.1 and
            abs(pos1[2] - pos2[2]) < 0.1):
            pairs.append((j1, j2))
            used.add(j1)
            used.add(j2)
            break

print(f"Pairs found: {len(pairs)}")

# Reconstruct
sc_fixed4 = slab4_2y.copy()
for keep_idx, move_idx in pairs:
    frac = ps_orig[keep_idx].frac_coords
    inv_frac = (inv_translation - frac) % 1.0
    inv_cart = ps_orig.lattice.get_cartesian_coords(inv_frac)
    sc_fixed4.positions[move_idx] = inv_cart

sym_f = np.array(sc_fixed4.get_chemical_symbols())
mg_f = sum(sym_f == 'Mg')
zn_f = sum(sym_f == 'Zn')

ps_fixed4 = AseAtomsAdaptor().get_structure(sc_fixed4)
slab_fixed4 = Slab(ps_fixed4.lattice, ps_fixed4.species, ps_fixed4.frac_coords,
    miller_index=(1,0,1), oriented_unit_cell=ps_fixed4, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

thick4 = sc_fixed4.get_positions()[:,2].max() - sc_fixed4.get_positions()[:,2].min()
area4 = np.linalg.norm(np.cross(sc_fixed4.cell[0], sc_fixed4.cell[1]))

print(f"\nAtoms: {len(sc_fixed4)}, Mg: {mg_f}, Zn: {zn_f}")
print(f"Stoichiometric: {'YES' if zn_f*4 == mg_f*7 else 'NO'}")
print(f"Symmetric: {slab_fixed4.is_symmetric()}")
print(f"Thickness: {thick4:.1f} A")
print(f"Area: {area4:.1f} A²")

Atoms: 880, Thickness: 45.4 A
Atomic layers: 288
  bot_rm=0, top_rm=3: 866 atoms, removed 0+14
SG = P2/m
Inversion: f -> [0.90063694 0.         0.99647266] - f

Removed: 14 atoms
Pairs found: 7

Atoms: 880, Mg: 320, Zn: 560
Stoichiometric: YES
Symmetric: True
Thickness: 45.6 A
Area: 348.8 A²


In [16]:
ps = AseAtomsAdaptor().get_structure(sc_fixed4)
ld = LammpsData.from_structure(ps, atom_style="atomic")
ld.write_file("slabs/mg4zn7_101_4layers_reconstructed_ase.data")

print(f"Saved: slabs/mg4zn7_101_4layers_reconstructed_ase.data")

Saved: slabs/mg4zn7_101_4layers_reconstructed_ase.data


In [17]:
#unrelaxed surface energy calculation
E_slab =  -1097.1706      # eV
E_bulk_per_fu =  -141.72792 / 10  # eV per formula unit 
n = 880 / 11                 # formula units in slab = 32
A = 348.8            # Ang^2

S_eV_per_ang2 = (E_slab - n * E_bulk_per_fu) / (2 * A)

# Convert to J/m^2 (1 eV/Ang^2 = 16.0218 J/m^2)
S_J_per_m2 = S_eV_per_ang2 * 16.0218

print(f"E_bulk per formula unit = {E_bulk_per_fu:.6f} eV")
print(f"n * E_bulk = {n * E_bulk_per_fu:.5f} eV")
print(f"E_slab - n*E_bulk = {E_slab - n * E_bulk_per_fu:.5f} eV")
print(f"S = {S_eV_per_ang2:.6f} eV/Ang^2")
print(f"S = {S_J_per_m2:.4f} J/m^2")

E_bulk per formula unit = -14.172792 eV
n * E_bulk = -1133.82336 eV
E_slab - n*E_bulk = 36.65276 eV
S = 0.052541 eV/Ang^2
S = 0.8418 J/m^2


In [18]:
#relaxed surface energy calculation
E_slab_relaxed =  -1105.33270915911  # eV
E_bulk_per_fu = -141.72792 / 10  # eV per formula unit
n = 880 / 11                      # 32 formula units
A = 348.8               # Ang^2

S_relaxed_eV = (E_slab_relaxed - n * E_bulk_per_fu) / (2 * A)
S_relaxed_Jm2 = S_relaxed_eV * 16.0218

print(f"E_slab_relaxed - n*E_bulk = {E_slab_relaxed - n * E_bulk_per_fu:.5f} eV")
print(f"S relaxed = {S_relaxed_eV:.6f} eV/Ang^2")
print(f"S relaxed = {S_relaxed_Jm2:.4f} J/m^2")

# Compare with unrelaxed
S_unrelaxed_Jm2 = 0.8418
print(f"\nUnrelaxed S = {S_unrelaxed_Jm2:.4f} J/m^2")
print(f"Relaxed S   = {S_relaxed_Jm2:.4f} J/m^2")
print(f"Relaxation energy = {S_unrelaxed_Jm2 - S_relaxed_Jm2:.4f} J/m^2")

E_slab_relaxed - n*E_bulk = 28.49065 eV
S relaxed = 0.040841 eV/Ang^2
S relaxed = 0.6543 J/m^2

Unrelaxed S = 0.8418 J/m^2
Relaxed S   = 0.6543 J/m^2
Relaxation energy = 0.1875 J/m^2
